# Diffusion Models: Learning to Generate Through Denoising

**Learning Objectives:**
- Understand how diffusion models work through progressive noise addition and removal
- Learn the forward diffusion process (adding noise)
- Learn the reverse diffusion process (learning to denoise)
- Implement a U-Net with time embeddings for denoising
- Train a Denoising Diffusion Probabilistic Model (DDPM)
- Compare sampling strategies (DDPM vs DDIM)
- Understand the connection to score-based models
- Compare diffusion models with VAEs and GANs

**What We'll Build:**
1. A complete forward diffusion process that progressively adds noise to images
2. A U-Net architecture with time embeddings to predict noise
3. A full DDPM training pipeline on MNIST
4. Sampling strategies for image generation
5. Visualizations showing the diffusion process in action

## Part 1: Introduction - Why Diffusion Models?

### The Evolution of Generative Models

You've learned about **VAEs** (explicit density models) and **GANs** (implicit density models). Each has trade-offs:

**VAEs:**
- ✅ Stable training
- ✅ Structured latent space
- ❌ Blurry images (MSE averaging)
- ❌ Limited sample quality

**GANs:**
- ✅ Sharp, realistic images
- ✅ High quality samples
- ❌ Training instability
- ❌ Mode collapse

### Enter Diffusion Models

**Diffusion models** combine the best of both worlds:
- ✅ Stable training (like VAEs)
- ✅ High quality, sharp images (like GANs)
- ✅ No mode collapse
- ✅ Principled probabilistic framework
- ❌ Slower sampling (requires many steps)

### The Core Idea

**Diffusion models work by:**
1. **Forward process**: Gradually add noise to data until it becomes pure Gaussian noise
2. **Reverse process**: Learn to remove noise step-by-step, generating data from noise

**Intuition**: It's easier to learn many small denoising steps than to generate complex data in one shot!

### Real-World Impact

Diffusion models power:
- **Stable Diffusion** (text-to-image generation)
- **DALL-E 2** (OpenAI's text-to-image model)
- **Midjourney** (AI art generation)
- **Imagen** (Google's text-to-image model)

This is arguably **THE most important generative modeling breakthrough** since GANs!

## Part 2: Setup and Imports

Let's import the necessary libraries and set up our environment.

In [ ]:


# Import from shared library

# Enable autoreload for development
%load_ext autoreload
%autoreload 2

# Set random seed for reproducibility
set_seed(42)

# Get device (prefer CPU for stability with some operations)
device = get_device(prefer_cpu=False)

print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

## Part 3: Load MNIST Dataset

We'll use MNIST to keep training fast while learning the core concepts. Diffusion models work on any image dataset!

In [ ]:
from torch.utils.data import DataLoaderfrom torchvision import datasets, transforms

In [ ]:
# Transform: convert to tensor and normalize to [-1, 1]
# Diffusion models typically work with data in [-1, 1] range
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))  # Maps [0, 1] to [-1, 1]
])

# Load datasets
train_dataset = datasets.MNIST(
    root='./tmp/data',
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    root='./tmp/data',
    train=False,
    download=True,
    transform=transform
)

# Create dataloaders
batch_size = 128
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Batch size: {batch_size}")
print(f"Training batches: {len(train_loader)}")

## Part 4: Visualizing Clean Data

Let's see what our clean data looks like before we start adding noise to it.

In [ ]:
from torchvision.utils import make_grid

In [ ]:
# Helper function to display images
def show_images(images, title="Images", nrow=8, figsize=(12, 6)):
    """Display a grid of images (assumes images in [-1, 1] range)."""
    # Denormalize from [-1, 1] to [0, 1]
    images = images * 0.5 + 0.5
    images = torch.clamp(images, 0, 1)  # Ensure in valid range
    
    grid = make_grid(images, nrow=nrow, padding=2)
    
    plt.figure(figsize=figsize)
    plt.imshow(grid.permute(1, 2, 0).cpu().numpy(), cmap='gray')
    plt.title(title)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

# Sample some images
sample_images, sample_labels = next(iter(train_loader))
show_images(sample_images[:64], "Clean MNIST Images")

## Part 5: The Forward Diffusion Process - Theory

### What is Forward Diffusion?

The **forward process** gradually destroys information by adding Gaussian noise over $T$ timesteps.

Starting with clean data $x_0$, we create a sequence of increasingly noisy versions:
$$x_0 \rightarrow x_1 \rightarrow x_2 \rightarrow \cdots \rightarrow x_T$$

At each step $t$, we add a small amount of Gaussian noise:
$$q(x_t | x_{t-1}) = \mathcal{N}(x_t; \sqrt{1 - \beta_t} x_{t-1}, \beta_t I)$$

Where:
- $\beta_t$ is the **noise schedule** (controls how much noise to add at step $t$)
- Typically $\beta_t$ increases over time: add more noise in later steps
- By step $T$ (e.g., $T=1000$), $x_T \approx \mathcal{N}(0, I)$ (pure noise)

### The Beautiful Math: Closed-Form Sampling

**Key insight**: We can sample $x_t$ directly from $x_0$ without computing all intermediate steps!

Define:
- $\alpha_t = 1 - \beta_t$
- $\bar{\alpha}_t = \prod_{s=1}^t \alpha_s$ (cumulative product)

Then:
$$q(x_t | x_0) = \mathcal{N}(x_t; \sqrt{\bar{\alpha}_t} x_0, (1 - \bar{\alpha}_t) I)$$

This means:
$$x_t = \sqrt{\bar{\alpha}_t} x_0 + \sqrt{1 - \bar{\alpha}_t} \epsilon, \quad \epsilon \sim \mathcal{N}(0, I)$$

**Why this matters:**
- Can sample any noisy version $x_t$ directly from $x_0$ in one step!
- Enables efficient training
- $\sqrt{\bar{\alpha}_t}$ controls the signal strength (clean data)
- $\sqrt{1 - \bar{\alpha}_t}$ controls the noise strength

## Part 6: Implementing the Noise Schedule

Let's implement the noise schedule that determines how much noise to add at each timestep.

In [ ]:
def linear_beta_schedule(timesteps, beta_start=0.0001, beta_end=0.02):
    """
    Linear schedule for beta values.
    
    Args:
        timesteps: Number of diffusion steps
        beta_start: Starting beta value (small noise)
        beta_end: Ending beta value (large noise)
    
    Returns:
        betas: Tensor of shape (timesteps,)
    """
    return torch.linspace(beta_start, beta_end, timesteps)

def cosine_beta_schedule(timesteps, s=0.008):
    """
    Cosine schedule from "Improved Denoising Diffusion Probabilistic Models".
    Better than linear schedule - adds noise more gradually.
    
    Args:
        timesteps: Number of diffusion steps
        s: Small offset to prevent beta_t from being too small near t=0
    
    Returns:
        betas: Tensor of shape (timesteps,)
    """
    steps = timesteps + 1
    x = torch.linspace(0, timesteps, steps)
    alphas_cumprod = torch.cos(((x / timesteps) + s) / (1 + s) * math.pi * 0.5) ** 2
    alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
    betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
    return torch.clip(betas, 0.0001, 0.9999)

# Create noise schedule
timesteps = 1000
betas = linear_beta_schedule(timesteps)

# Compute alphas and alpha_bar (cumulative product)
alphas = 1.0 - betas
alphas_cumprod = torch.cumprod(alphas, dim=0)
alphas_cumprod_prev = F.pad(alphas_cumprod[:-1], (1, 0), value=1.0)

# Compute square roots for sampling (will use these often)
sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod)
sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - alphas_cumprod)

print(f"Number of timesteps: {timesteps}")
print(f"Beta range: [{betas.min():.6f}, {betas.max():.6f}]")
print(f"Alpha_bar at t=0: {alphas_cumprod[0]:.6f}")
print(f"Alpha_bar at t={timesteps//2}: {alphas_cumprod[timesteps//2]:.6f}")
print(f"Alpha_bar at t={timesteps-1}: {alphas_cumprod[-1]:.6f}")

## Part 7: Visualizing the Noise Schedule

Let's visualize how the noise schedule evolves over timesteps.

In [ ]:
# Plot the noise schedule
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Plot 1: Beta schedule
axes[0].plot(betas.numpy())
axes[0].set_xlabel('Timestep t')
axes[0].set_ylabel('Beta_t')
axes[0].set_title('Noise Schedule (Beta)')
axes[0].grid(True, alpha=0.3)

# Plot 2: Alpha_bar (cumulative product)
axes[1].plot(alphas_cumprod.numpy())
axes[1].set_xlabel('Timestep t')
axes[1].set_ylabel('Alpha_bar_t')
axes[1].set_title('Cumulative Product (Alpha_bar)')
axes[1].grid(True, alpha=0.3)

# Plot 3: Signal vs Noise ratio
signal_ratio = sqrt_alphas_cumprod.numpy()
noise_ratio = sqrt_one_minus_alphas_cumprod.numpy()
axes[2].plot(signal_ratio, label='Signal: sqrt(alpha_bar_t)', alpha=0.7)
axes[2].plot(noise_ratio, label='Noise: sqrt(1 - alpha_bar_t)', alpha=0.7)
axes[2].set_xlabel('Timestep t')
axes[2].set_ylabel('Ratio')
axes[2].set_title('Signal vs Noise Ratio')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nKey observations:")
print("1. Beta increases linearly (more noise added in later steps)")
print("2. Alpha_bar decreases (signal strength reduces over time)")
print("3. Signal ratio decreases while noise ratio increases")
print("4. By t=1000, signal ≈ 0 and noise ≈ 1 (pure Gaussian noise)")

## Part 8: Implementing Forward Diffusion

Now let's implement the function to add noise to images at any timestep $t$.

In [ ]:
def extract(a, t, x_shape):
    """
    Extract coefficients from a based on timestep t.
    
    Args:
        a: Coefficient tensor (e.g., sqrt_alphas_cumprod)
        t: Timestep tensor of shape (batch_size,)
        x_shape: Shape of x for broadcasting
    
    Returns:
        Extracted coefficients reshaped for broadcasting
    """
    batch_size = t.shape[0]
    out = a.gather(-1, t.cpu())
    return out.reshape(batch_size, *((1,) * (len(x_shape) - 1))).to(t.device)

def forward_diffusion(x_0, t, sqrt_alphas_cumprod, sqrt_one_minus_alphas_cumprod, noise=None):
    """
    Apply forward diffusion to add noise to x_0 at timestep t.
    
    Uses the formula: x_t = sqrt(alpha_bar_t) * x_0 + sqrt(1 - alpha_bar_t) * epsilon
    
    Args:
        x_0: Clean images, shape (batch_size, channels, height, width)
        t: Timesteps, shape (batch_size,)
        sqrt_alphas_cumprod: Precomputed sqrt(alpha_bar)
        sqrt_one_minus_alphas_cumprod: Precomputed sqrt(1 - alpha_bar)
        noise: Optional noise tensor (will be sampled if None)
    
    Returns:
        x_t: Noisy images at timestep t
        noise: The noise that was added
    """
    if noise is None:
        noise = torch.randn_like(x_0)
    
    # Extract coefficients for timestep t
    sqrt_alpha_bar_t = extract(sqrt_alphas_cumprod, t, x_0.shape)
    sqrt_one_minus_alpha_bar_t = extract(sqrt_one_minus_alphas_cumprod, t, x_0.shape)
    
    # Apply forward diffusion formula
    x_t = sqrt_alpha_bar_t * x_0 + sqrt_one_minus_alpha_bar_t * noise
    
    return x_t, noise

# Test forward diffusion
test_image = sample_images[0:1].to(device)
test_timesteps = torch.tensor([0, 100, 250, 500, 750, 999]).to(device)

print("Testing forward diffusion at different timesteps:")
for t in test_timesteps:
    t_batch = t.unsqueeze(0)
    x_t, noise = forward_diffusion(
        test_image, 
        t_batch, 
        sqrt_alphas_cumprod.to(device), 
        sqrt_one_minus_alphas_cumprod.to(device)
    )
    print(f"  t={t.item():4d}: x_t range [{x_t.min():.2f}, {x_t.max():.2f}]")

## Part 9: Visualizing the Forward Diffusion Process

Let's visualize how an image gradually becomes noise through the forward diffusion process.

In [ ]:
# Select one image and apply forward diffusion at different timesteps
image = sample_images[0:1].to(device)
timesteps_to_show = [0, 50, 100, 200, 400, 600, 800, 999]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

# Use same noise for all timesteps to show progression
noise = torch.randn_like(image)

for idx, t in enumerate(timesteps_to_show):
    t_tensor = torch.tensor([t]).to(device)
    x_t, _ = forward_diffusion(
        image, 
        t_tensor, 
        sqrt_alphas_cumprod.to(device), 
        sqrt_one_minus_alphas_cumprod.to(device),
        noise=noise
    )
    
    # Denormalize and display
    img_display = (x_t[0, 0].cpu() * 0.5 + 0.5).clamp(0, 1)
    axes[idx].imshow(img_display, cmap='gray')
    axes[idx].set_title(f't = {t}')
    axes[idx].axis('off')

plt.suptitle('Forward Diffusion: Clean Image → Pure Noise', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Notice how:")
print("- At t=0: Clean digit is perfectly visible")
print("- At t=50-200: Digit is recognizable but noisy")
print("- At t=400-600: Digit shape is barely visible")
print("- At t=999: Pure Gaussian noise, no trace of original image")

## Part 10: The Reverse Diffusion Process - Theory

### Learning to Denoise

The **reverse process** learns to remove noise step-by-step, going from $x_T$ (pure noise) back to $x_0$ (clean data):
$$x_T \rightarrow x_{T-1} \rightarrow \cdots \rightarrow x_1 \rightarrow x_0$$

If we knew the reverse conditional $q(x_{t-1} | x_t)$, we could sample by:
1. Sample $x_T \sim \mathcal{N}(0, I)$
2. For $t = T, T-1, \ldots, 1$: sample $x_{t-1} \sim q(x_{t-1} | x_t)$
3. Return $x_0$

**Problem**: $q(x_{t-1} | x_t)$ is intractable (requires knowing the full data distribution)!

### Solution: Learn a Neural Network

We approximate the reverse process with a neural network:
$$p_\theta(x_{t-1} | x_t) = \mathcal{N}(x_{t-1}; \mu_\theta(x_t, t), \Sigma_\theta(x_t, t))$$

Where:
- $\mu_\theta$ is a neural network that predicts the mean
- $\Sigma_\theta$ is the variance (often fixed)

### What Should the Network Predict?

**Option 1**: Predict the mean $\mu_\theta$ directly
**Option 2**: Predict the noise $\epsilon_\theta$ (this works better!)

The DDPM paper shows we can parameterize the mean as:
$$\mu_\theta(x_t, t) = \frac{1}{\sqrt{\alpha_t}} \left( x_t - \frac{\beta_t}{\sqrt{1 - \bar{\alpha}_t}} \epsilon_\theta(x_t, t) \right)$$

So instead of predicting $x_{t-1}$ directly, we:
1. Train a network $\epsilon_\theta(x_t, t)$ to predict the noise $\epsilon$
2. Use the predicted noise to compute the mean $\mu_\theta$
3. Sample $x_{t-1}$ from the Gaussian distribution

### Training Objective (Simplified)

The loss is simply:
$$L = \mathbb{E}_{t, x_0, \epsilon} \left[ \| \epsilon - \epsilon_\theta(x_t, t) \|^2 \right]$$

In words:
1. Sample a random timestep $t$
2. Sample clean data $x_0$
3. Sample noise $\epsilon$
4. Create noisy image $x_t$ using forward diffusion
5. Train network to predict the noise

**This is just a noise prediction task!** Much simpler than it sounds.

## Part 11: U-Net Architecture with Time Embeddings

### Why U-Net?

Diffusion models use **U-Net** architecture because:
1. **Skip connections**: Preserve fine details from input
2. **Multi-scale processing**: Encoder-decoder handles different levels of detail
3. **Proven for image-to-image tasks**: Originally designed for image segmentation

### Time Embeddings

The network needs to know **which timestep** $t$ it's denoising. We use **sinusoidal position embeddings** (like in Transformers) to encode time:

$$\text{PE}(t, 2i) = \sin(t / 10000^{2i/d})$$
$$\text{PE}(t, 2i+1) = \cos(t / 10000^{2i/d})$$

These embeddings are injected into the network at each layer.

## Part 12: Implementing Time Embeddings

Let's implement sinusoidal time embeddings for the diffusion timesteps.

In [ ]:
class SinusoidalPositionEmbeddings(nn.Module):
    """
    Sinusoidal position embeddings for timestep encoding.
    Same as used in Transformers for position encoding.
    """
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, time):
        """
        Args:
            time: Tensor of shape (batch_size,) with timestep values
        
        Returns:
            Embeddings of shape (batch_size, dim)
        """
        device = time.device
        half_dim = self.dim // 2
        embeddings = math.log(10000) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=device) * -embeddings)
        embeddings = time[:, None] * embeddings[None, :]
        embeddings = torch.cat((embeddings.sin(), embeddings.cos()), dim=-1)
        return embeddings

# Test time embeddings
time_emb = SinusoidalPositionEmbeddings(dim=128)
test_times = torch.tensor([0, 250, 500, 750, 999])
test_embeddings = time_emb(test_times)

print(f"Time embedding shape: {test_embeddings.shape}")
print(f"Embedding statistics:")
print(f"  Mean: {test_embeddings.mean():.4f}")
print(f"  Std: {test_embeddings.std():.4f}")
print(f"  Range: [{test_embeddings.min():.4f}, {test_embeddings.max():.4f}]")

## Part 13: Building the U-Net Components

Let's build the core components of our U-Net: residual blocks with time conditioning.

In [ ]:
class ResidualBlock(nn.Module):
    """
    Residual block with time embedding conditioning.
    
    Architecture:
    - Two conv layers with GroupNorm
    - Time embedding is projected and added to features
    - Skip connection from input to output
    """
    def __init__(self, in_channels, out_channels, time_emb_dim, dropout=0.1):
        super().__init__()
        
        # First conv block
        self.conv1 = nn.Sequential(
            nn.GroupNorm(8, in_channels),
            nn.SiLU(),  # SiLU (Swish) activation works well for diffusion models
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        )
        
        # Time embedding projection
        self.time_mlp = nn.Sequential(
            nn.SiLU(),
            nn.Linear(time_emb_dim, out_channels)
        )
        
        # Second conv block
        self.conv2 = nn.Sequential(
            nn.GroupNorm(8, out_channels),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        )
        
        # Residual connection (1x1 conv if channels change)
        self.residual = nn.Conv2d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()
    
    def forward(self, x, time_emb):
        """
        Args:
            x: Input features (batch_size, in_channels, height, width)
            time_emb: Time embeddings (batch_size, time_emb_dim)
        
        Returns:
            Output features (batch_size, out_channels, height, width)
        """
        residual = self.residual(x)
        
        # First conv
        h = self.conv1(x)
        
        # Add time embedding (broadcast across spatial dimensions)
        time_emb = self.time_mlp(time_emb)
        h = h + time_emb[:, :, None, None]
        
        # Second conv
        h = self.conv2(h)
        
        return h + residual

# Test residual block
test_block = ResidualBlock(in_channels=64, out_channels=128, time_emb_dim=128).to(device)
test_x = torch.randn(4, 64, 28, 28).to(device)
test_t_emb = torch.randn(4, 128).to(device)
test_out = test_block(test_x, test_t_emb)

print(f"Residual block test:")
print(f"  Input shape: {test_x.shape}")
print(f"  Output shape: {test_out.shape}")
print(f"  Parameters: {sum(p.numel() for p in test_block.parameters()):,}")

## Part 14: Attention Mechanism for U-Net

We'll add self-attention layers to help the model capture long-range dependencies in the image.

In [ ]:
class AttentionBlock(nn.Module):
    """
    Self-attention block for spatial features.
    Helps model capture long-range dependencies.
    """
    def __init__(self, channels, num_heads=4):
        super().__init__()
        self.channels = channels
        self.num_heads = num_heads
        
        self.norm = nn.GroupNorm(8, channels)
        self.qkv = nn.Conv2d(channels, channels * 3, kernel_size=1)
        self.proj = nn.Conv2d(channels, channels, kernel_size=1)
    
    def forward(self, x):
        """
        Args:
            x: Input features (batch_size, channels, height, width)
        
        Returns:
            Output features with same shape as input
        """
        batch_size, channels, height, width = x.shape
        residual = x
        
        # Normalize
        x = self.norm(x)
        
        # Compute Q, K, V
        qkv = self.qkv(x)  # (B, 3C, H, W)
        qkv = qkv.reshape(batch_size, 3, self.num_heads, channels // self.num_heads, height * width)
        qkv = qkv.permute(1, 0, 2, 4, 3)  # (3, B, num_heads, H*W, C//num_heads)
        q, k, v = qkv[0], qkv[1], qkv[2]
        
        # Scaled dot-product attention
        scale = (channels // self.num_heads) ** -0.5
        attn = torch.softmax(torch.matmul(q, k.transpose(-2, -1)) * scale, dim=-1)
        
        # Apply attention to values
        out = torch.matmul(attn, v)  # (B, num_heads, H*W, C//num_heads)
        out = out.permute(0, 1, 3, 2)  # (B, num_heads, C//num_heads, H*W)
        out = out.reshape(batch_size, channels, height, width)
        
        # Project and add residual
        out = self.proj(out)
        return out + residual

# Test attention block
test_attn = AttentionBlock(channels=128, num_heads=4).to(device)
test_x = torch.randn(4, 128, 14, 14).to(device)
test_out = test_attn(test_x)

print(f"Attention block test:")
print(f"  Input shape: {test_x.shape}")
print(f"  Output shape: {test_out.shape}")
print(f"  Parameters: {sum(p.numel() for p in test_attn.parameters()):,}")

## Part 15: Complete U-Net Implementation

Now let's build the complete U-Net with encoder, bottleneck, and decoder.

In [ ]:
class UNet(nn.Module):
    """
    U-Net for noise prediction in diffusion models.
    
    Architecture:
    - Encoder: Downsample with residual blocks
    - Bottleneck: Process at lowest resolution with attention
    - Decoder: Upsample with residual blocks and skip connections
    - Time embeddings: Injected at each residual block
    """
    def __init__(
        self,
        in_channels=1,
        model_channels=64,
        out_channels=1,
        num_res_blocks=2,
        attention_resolutions=(2,),  # Apply attention at these resolutions
        dropout=0.1,
        channel_mult=(1, 2, 4),  # Channel multipliers at each resolution
        time_emb_dim=128
    ):
        super().__init__()
        
        self.in_channels = in_channels
        self.model_channels = model_channels
        self.out_channels = out_channels
        self.num_res_blocks = num_res_blocks
        self.attention_resolutions = attention_resolutions
        
        # Time embedding
        self.time_embed = nn.Sequential(
            SinusoidalPositionEmbeddings(model_channels),
            nn.Linear(model_channels, time_emb_dim),
            nn.SiLU(),
            nn.Linear(time_emb_dim, time_emb_dim),
        )
        
        # Initial convolution
        self.conv_in = nn.Conv2d(in_channels, model_channels, kernel_size=3, padding=1)
        
        # Encoder (downsampling)
        self.encoder = nn.ModuleList()
        channels = [model_channels]
        current_channels = model_channels
        
        for level, mult in enumerate(channel_mult):
            out_ch = model_channels * mult
            
            for _ in range(num_res_blocks):
                layers = [
                    ResidualBlock(current_channels, out_ch, time_emb_dim, dropout)
                ]
                current_channels = out_ch
                
                # Add attention at specified resolutions
                if level in attention_resolutions:
                    layers.append(AttentionBlock(current_channels))
                
                self.encoder.append(nn.ModuleList(layers))
                channels.append(current_channels)
            
            # Downsample (except at last level)
            if level != len(channel_mult) - 1:
                self.encoder.append(nn.ModuleList([nn.Conv2d(current_channels, current_channels, 3, stride=2, padding=1)]))
                channels.append(current_channels)
        
        # Bottleneck
        self.bottleneck = nn.ModuleList([
            ResidualBlock(current_channels, current_channels, time_emb_dim, dropout),
            AttentionBlock(current_channels),
            ResidualBlock(current_channels, current_channels, time_emb_dim, dropout),
        ])
        
        # Decoder (upsampling)
        self.decoder = nn.ModuleList()
        
        for level, mult in list(enumerate(channel_mult))[::-1]:
            out_ch = model_channels * mult
            
            for i in range(num_res_blocks + 1):
                # Pop skip connection channel count
                skip_channels = channels.pop()
                
                layers = [
                    ResidualBlock(current_channels + skip_channels, out_ch, time_emb_dim, dropout)
                ]
                current_channels = out_ch
                
                # Add attention at specified resolutions
                if level in attention_resolutions:
                    layers.append(AttentionBlock(current_channels))
                
                # Upsample on last block of each resolution (except first)
                if level != 0 and i == num_res_blocks:
                    layers.append(nn.ConvTranspose2d(current_channels, current_channels, 4, stride=2, padding=1))
                
                self.decoder.append(nn.ModuleList(layers))
        
        # Output
        self.conv_out = nn.Sequential(
            nn.GroupNorm(8, current_channels),
            nn.SiLU(),
            nn.Conv2d(current_channels, out_channels, kernel_size=3, padding=1),
        )
    
    def forward(self, x, time):
        """
        Args:
            x: Noisy images (batch_size, channels, height, width)
            time: Timesteps (batch_size,)
        
        Returns:
            Predicted noise (batch_size, channels, height, width)
        """
        # Time embedding
        time_emb = self.time_embed(time)
        
        # Initial convolution
        h = self.conv_in(x)
        
        # Store skip connections
        skip_connections = [h]
        
        # Encoder
        for module in self.encoder:
            for layer in module:
                if isinstance(layer, ResidualBlock):
                    h = layer(h, time_emb)
                else:
                    h = layer(h)
            skip_connections.append(h)
        
        # Bottleneck
        for layer in self.bottleneck:
            if isinstance(layer, ResidualBlock):
                h = layer(h, time_emb)
            else:
                h = layer(h)
        
        # Decoder
        for module in self.decoder:
            # Concatenate skip connection
            skip = skip_connections.pop()
            h = torch.cat([h, skip], dim=1)
            
            for layer in module:
                if isinstance(layer, ResidualBlock):
                    h = layer(h, time_emb)
                else:
                    h = layer(h)
        
        # Output
        return self.conv_out(h)

# Create model
model = UNet(
    in_channels=1,
    model_channels=64,
    out_channels=1,
    num_res_blocks=2,
    attention_resolutions=(1,),
    channel_mult=(1, 2, 4),
    time_emb_dim=128
).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"\nU-Net Model:")
print(f"  Total parameters: {total_params:,}")

# Test forward pass
test_x = torch.randn(4, 1, 28, 28).to(device)
test_t = torch.randint(0, 1000, (4,)).to(device)
test_noise_pred = model(test_x, test_t)

print(f"  Input shape: {test_x.shape}")
print(f"  Time shape: {test_t.shape}")
print(f"  Output (predicted noise) shape: {test_noise_pred.shape}")

## Part 16: Training Loss - Noise Prediction

The training objective is beautifully simple: predict the noise that was added!

In [ ]:
def diffusion_loss(model, x_0, sqrt_alphas_cumprod, sqrt_one_minus_alphas_cumprod, timesteps):
    """
    Compute the diffusion model training loss.
    
    Algorithm:
    1. Sample random timesteps t
    2. Sample random noise epsilon
    3. Create noisy images x_t using forward diffusion
    4. Predict noise using model
    5. Compute MSE between true noise and predicted noise
    
    Args:
        model: Noise prediction model
        x_0: Clean images (batch_size, channels, height, width)
        sqrt_alphas_cumprod: Precomputed sqrt(alpha_bar)
        sqrt_one_minus_alphas_cumprod: Precomputed sqrt(1 - alpha_bar)
        timesteps: Number of diffusion timesteps
    
    Returns:
        loss: Mean squared error between true and predicted noise
    """
    batch_size = x_0.shape[0]
    
    # Sample random timesteps
    t = torch.randint(0, timesteps, (batch_size,), device=x_0.device).long()
    
    # Sample noise
    noise = torch.randn_like(x_0)
    
    # Create noisy images
    x_t, _ = forward_diffusion(x_0, t, sqrt_alphas_cumprod, sqrt_one_minus_alphas_cumprod, noise=noise)
    
    # Predict noise
    noise_pred = model(x_t, t)
    
    # Compute loss (simple MSE)
    loss = F.mse_loss(noise_pred, noise)
    
    return loss

# Test loss computation
test_images = sample_images[:4].to(device)
test_loss = diffusion_loss(
    model, 
    test_images, 
    sqrt_alphas_cumprod.to(device), 
    sqrt_one_minus_alphas_cumprod.to(device),
    timesteps
)

print(f"Test loss: {test_loss.item():.4f}")
print("\nLoss interpretation:")
print("  - Random model: ~1.0 (predicting mean of Gaussian)")
  "  - Perfect model: 0.0 (exactly predicting noise)")
print("  - Our untrained model: ~{:.2f}".format(test_loss.item()))

## Part 17: Sampling Algorithm - DDPM

To generate images, we reverse the diffusion process step by step.

In [ ]:
from tqdm.auto import tqdm

In [ ]:
@torch.no_grad()
def ddpm_sample(model, image_size, batch_size, timesteps, betas, alphas, alphas_cumprod, sqrt_alphas_cumprod, sqrt_one_minus_alphas_cumprod, device):
    """
    Generate samples using DDPM sampling (reverse diffusion).
    
    Algorithm:
    1. Start with pure noise x_T ~ N(0, I)
    2. For t = T, T-1, ..., 1:
        a. Predict noise epsilon_theta(x_t, t)
        b. Compute mean of p(x_{t-1} | x_t)
        c. Sample x_{t-1} from N(mean, variance)
    3. Return x_0
    
    Args:
        model: Trained noise prediction model
        image_size: Tuple of (channels, height, width)
        batch_size: Number of images to generate
        ... (various precomputed diffusion constants)
    
    Returns:
        Generated images (batch_size, channels, height, width)
    """
    model.eval()
    
    # Start from pure noise
    img = torch.randn(batch_size, *image_size, device=device)
    
    # Reverse diffusion process
    for t in tqdm(reversed(range(timesteps)), total=timesteps, desc="Sampling"):
        # Create batch of timesteps
        t_batch = torch.full((batch_size,), t, device=device, dtype=torch.long)
        
        # Predict noise
        predicted_noise = model(img, t_batch)
        
        # Extract coefficients
        alpha_t = extract(alphas, t_batch, img.shape)
        alpha_bar_t = extract(alphas_cumprod, t_batch, img.shape)
        beta_t = extract(betas, t_batch, img.shape)
        sqrt_one_minus_alpha_bar_t = extract(sqrt_one_minus_alphas_cumprod, t_batch, img.shape)
        
        # Compute mean of p(x_{t-1} | x_t)
        # Using formula: mean = (1/sqrt(alpha_t)) * (x_t - (beta_t / sqrt(1 - alpha_bar_t)) * epsilon_theta)
        mean = (1 / torch.sqrt(alpha_t)) * (img - (beta_t / sqrt_one_minus_alpha_bar_t) * predicted_noise)
        
        if t > 0:
            # Add noise (except for last step)
            noise = torch.randn_like(img)
            
            # Compute variance
            # Using simplified formula: variance = beta_t
            variance = beta_t
            
            # Sample x_{t-1}
            img = mean + torch.sqrt(variance) * noise
        else:
            # Final step: no noise
            img = mean
    
    return img

print("DDPM sampling function defined")
print("This will generate images by iteratively denoising pure noise")

## Part 18: Training Loop

Now let's train the diffusion model! This will take some time but watch the progress.

In [ ]:
# Training configuration
num_epochs = 20
learning_rate = 2e-4

# Move constants to device
betas = betas.to(device)
alphas = alphas.to(device)
alphas_cumprod = alphas_cumprod.to(device)
alphas_cumprod_prev = alphas_cumprod_prev.to(device)
sqrt_alphas_cumprod = sqrt_alphas_cumprod.to(device)
sqrt_one_minus_alphas_cumprod = sqrt_one_minus_alphas_cumprod.to(device)

# Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

# Training history
history = {'train_loss': []}

print(f"Training diffusion model for {num_epochs} epochs...\n")

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0.0
    
    for batch_idx, (images, _) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=False)):
        images = images.to(device)
        
        # Compute loss
        loss = diffusion_loss(
            model, 
            images, 
            sqrt_alphas_cumprod, 
            sqrt_one_minus_alphas_cumprod,
            timesteps
        )
        
        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    # Average loss
    avg_loss = epoch_loss / len(train_loader)
    history['train_loss'].append(avg_loss)
    
    print(f"Epoch {epoch+1}/{num_epochs} - Loss: {avg_loss:.4f}")
    
    # Generate samples every 5 epochs
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"  Generating samples...")
        samples = ddpm_sample(
            model, 
            image_size=(1, 28, 28),
            batch_size=64,
            timesteps=timesteps,
            betas=betas,
            alphas=alphas,
            alphas_cumprod=alphas_cumprod,
            sqrt_alphas_cumprod=sqrt_alphas_cumprod,
            sqrt_one_minus_alphas_cumprod=sqrt_one_minus_alphas_cumprod,
            device=device
        )
        
        # Show generated samples
        show_images(samples, f"Generated Samples - Epoch {epoch+1}")
        
        # Plot training curve
        plt.figure(figsize=(8, 4))
        plt.plot(history['train_loss'])
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.title('Training Loss')
        plt.grid(True, alpha=0.3)
        plt.show()

print("\nTraining complete!")

## Part 19: Visualizing the Reverse Diffusion Process

Let's visualize how the model gradually denoises pure noise into a digit!

In [ ]:
@torch.no_grad()
def visualize_reverse_diffusion(model, image_size, timesteps, betas, alphas, alphas_cumprod, sqrt_alphas_cumprod, sqrt_one_minus_alphas_cumprod, device, steps_to_show=8):
    """
    Generate one sample and save intermediate denoising steps for visualization.
    """
    model.eval()
    
    # Start from pure noise
    img = torch.randn(1, *image_size, device=device)
    
    # Track intermediate steps
    intermediates = []
    save_steps = torch.linspace(0, timesteps-1, steps_to_show).long().tolist()
    save_steps = sorted(save_steps, reverse=True)
    
    # Reverse diffusion
    for t in reversed(range(timesteps)):
        t_batch = torch.full((1,), t, device=device, dtype=torch.long)
        
        # Predict noise
        predicted_noise = model(img, t_batch)
        
        # Extract coefficients
        alpha_t = extract(alphas, t_batch, img.shape)
        beta_t = extract(betas, t_batch, img.shape)
        sqrt_one_minus_alpha_bar_t = extract(sqrt_one_minus_alphas_cumprod, t_batch, img.shape)
        
        # Compute mean
        mean = (1 / torch.sqrt(alpha_t)) * (img - (beta_t / sqrt_one_minus_alpha_bar_t) * predicted_noise)
        
        if t > 0:
            noise = torch.randn_like(img)
            variance = beta_t
            img = mean + torch.sqrt(variance) * noise
        else:
            img = mean
        
        # Save intermediate steps
        if t in save_steps:
            intermediates.append(img.clone())
    
    return torch.cat(intermediates, dim=0)

# Visualize the denoising process
denoising_steps = visualize_reverse_diffusion(
    model,
    image_size=(1, 28, 28),
    timesteps=timesteps,
    betas=betas,
    alphas=alphas,
    alphas_cumprod=alphas_cumprod,
    sqrt_alphas_cumprod=sqrt_alphas_cumprod,
    sqrt_one_minus_alphas_cumprod=sqrt_one_minus_alphas_cumprod,
    device=device,
    steps_to_show=8
)

show_images(denoising_steps, "Reverse Diffusion: Pure Noise → Clean Image", nrow=8, figsize=(16, 3))

print("Notice how the image gradually emerges from noise!")
print("Early steps: Remove large-scale noise structure")
print("Later steps: Refine fine details and edges")

## Part 20: DDIM Sampling - Faster Generation

### The Problem with DDPM

DDPM requires **1000 steps** to generate one image - very slow!

### DDIM: Denoising Diffusion Implicit Models

**Key insight**: We don't need to reverse every single step! We can skip timesteps.

DDIM uses a **deterministic** sampling process (no added noise) that allows:
- Generate in 50-100 steps instead of 1000
- **10-20x faster** sampling
- Similar quality to DDPM

The DDIM update rule:
$$x_{t-\tau} = \sqrt{\bar{\alpha}_{t-\tau}} \cdot \underbrace{\frac{x_t - \sqrt{1-\bar{\alpha}_t} \cdot \epsilon_\theta(x_t, t)}{\sqrt{\bar{\alpha}_t}}}_{\text{predicted } x_0} + \sqrt{1 - \bar{\alpha}_{t-\tau}} \cdot \epsilon_\theta(x_t, t)$$

Where $\tau$ is the skip size (e.g., skip 20 steps at a time).

## Part 21: Implementing DDIM Sampling

Let's implement DDIM for faster sampling.

In [ ]:
@torch.no_grad()
def ddim_sample(model, image_size, batch_size, timesteps, alphas_cumprod, sqrt_alphas_cumprod, sqrt_one_minus_alphas_cumprod, device, ddim_steps=50):
    """
    Generate samples using DDIM (faster than DDPM).
    
    DDIM uses a deterministic, non-Markovian process that allows
    skipping timesteps for faster sampling.
    
    Args:
        ddim_steps: Number of denoising steps (much less than timesteps)
    """
    model.eval()
    
    # Create subsequence of timesteps to use
    skip = timesteps // ddim_steps
    seq = range(0, timesteps, skip)
    seq = list(reversed(list(seq)))
    
    # Start from pure noise
    img = torch.randn(batch_size, *image_size, device=device)
    
    # Reverse diffusion with skipped timesteps
    for i, t in enumerate(tqdm(seq, desc=f"DDIM Sampling ({ddim_steps} steps)")):
        t_batch = torch.full((batch_size,), t, device=device, dtype=torch.long)
        
        # Predict noise
        predicted_noise = model(img, t_batch)
        
        # Get alpha values
        alpha_bar_t = extract(alphas_cumprod, t_batch, img.shape)
        
        # Predict x_0
        pred_x0 = (img - torch.sqrt(1 - alpha_bar_t) * predicted_noise) / torch.sqrt(alpha_bar_t)
        pred_x0 = torch.clamp(pred_x0, -1, 1)
        
        if i < len(seq) - 1:
            # Get next timestep
            t_next = seq[i + 1]
            t_next_batch = torch.full((batch_size,), t_next, device=device, dtype=torch.long)
            alpha_bar_t_next = extract(alphas_cumprod, t_next_batch, img.shape)
            
            # DDIM update (deterministic)
            img = torch.sqrt(alpha_bar_t_next) * pred_x0 + torch.sqrt(1 - alpha_bar_t_next) * predicted_noise
        else:
            # Final step
            img = pred_x0
    
    return img

# Compare DDPM vs DDIM sampling time
import time

print("Comparing sampling speeds...\n")

# DDIM (50 steps)
start = time.time()
ddim_samples = ddim_sample(
    model,
    image_size=(1, 28, 28),
    batch_size=16,
    timesteps=timesteps,
    alphas_cumprod=alphas_cumprod,
    sqrt_alphas_cumprod=sqrt_alphas_cumprod,
    sqrt_one_minus_alphas_cumprod=sqrt_one_minus_alphas_cumprod,
    device=device,
    ddim_steps=50
)
ddim_time = time.time() - start

print(f"\nDDIM (50 steps): {ddim_time:.2f}s")
print(f"DDPM would take ~{ddim_time * 20:.2f}s (1000 steps)")
print(f"Speedup: ~20x faster!")

show_images(ddim_samples, "DDIM Generated Samples (50 steps)")

## Part 22: Connection to Score-Based Models

### The Score Function

Diffusion models are closely related to **score-based generative models**.

The **score** is the gradient of the log probability:
$$s_\theta(x, t) = \nabla_x \log p_t(x)$$

This tells us the direction to move in data space to increase probability.

### The Connection

It turns out that predicting noise $\epsilon_\theta$ is equivalent to predicting the score!

Specifically:
$$\nabla_x \log p_t(x_t) = -\frac{1}{\sqrt{1 - \bar{\alpha}_t}} \epsilon_\theta(x_t, t)$$

So when we train a diffusion model to predict noise, we're actually learning the score function!

### Why This Matters

**Score-based view provides intuition:**
- The model learns the gradient field pointing toward high-probability regions
- Sampling follows this gradient field from noise to data
- This is called **Langevin dynamics** in physics

**Different perspectives, same model:**
- **DDPM**: Learn to predict and remove noise
- **Score matching**: Learn gradients of the data distribution
- **Stochastic differential equations (SDEs)**: Continuous-time diffusion

All three frameworks describe the same underlying process!

## Part 23: Comparing VAE vs GAN vs Diffusion

Let's generate samples with our trained diffusion model and compare with what we know about VAEs and GANs.

In [ ]:
# Generate high-quality samples with diffusion model
diffusion_samples = ddim_sample(
    model,
    image_size=(1, 28, 28),
    batch_size=64,
    timesteps=timesteps,
    alphas_cumprod=alphas_cumprod,
    sqrt_alphas_cumprod=sqrt_alphas_cumprod,
    sqrt_one_minus_alphas_cumprod=sqrt_one_minus_alphas_cumprod,
    device=device,
    ddim_steps=100  # More steps for higher quality
)

show_images(diffusion_samples, "Diffusion Model Generated Samples")

# Show real data for comparison
real_samples, _ = next(iter(test_loader))
show_images(real_samples[:64], "Real MNIST Samples")

## Part 24: Comparison Table - VAE vs GAN vs Diffusion

### Comprehensive Comparison

| Aspect | VAE | GAN | Diffusion |
|--------|-----|-----|----------|
| **Sample Quality** | Blurry, averaged | Sharp, realistic | Sharp, high-quality |
| **Training Stability** | ✅ Very stable | ❌ Difficult, unstable | ✅ Very stable |
| **Mode Coverage** | ✅ Good coverage | ❌ Mode collapse risk | ✅ Excellent coverage |
| **Sampling Speed** | ✅ Fast (1 forward pass) | ✅ Fast (1 forward pass) | ❌ Slow (50-1000 steps) |
| **Training Speed** | ✅ Fast | ✅ Fast | ⚠️ Moderate |
| **Latent Space** | ✅ Structured, interpretable | ⚠️ Less structured | ⚠️ Noisy latent (less useful) |
| **Likelihood** | ✅ Can compute | ❌ Cannot compute | ⚠️ Expensive to compute |
| **Architecture** | Encoder + Decoder | Generator + Discriminator | U-Net with time embeddings |
| **Loss Function** | ELBO (reconstruction + KL) | Adversarial loss | Simple MSE (noise prediction) |
| **Hyperparameter Tuning** | ✅ Relatively easy | ❌ Very sensitive | ✅ Relatively easy |
| **Theoretical Foundation** | Probabilistic (variational inference) | Game theory (minimax) | Probabilistic (score matching) |

### When to Use Each?

**Use VAE when:**
- Need fast sampling
- Want interpretable latent space
- Need likelihood estimates
- Working with limited compute
- **Examples**: Anomaly detection, compression, representation learning

**Use GAN when:**
- Need highest quality with fast sampling
- Can afford extensive tuning
- Have expert knowledge to stabilize training
- Speed is critical in deployment
- **Examples**: Real-time generation, style transfer

**Use Diffusion when:**
- Need highest quality (state-of-the-art)
- Want stable training
- Can afford slow sampling
- Need good mode coverage
- **Examples**: Text-to-image, image editing, high-quality generation

### The Current State (2024-2025)

**Diffusion models are winning for high-quality generation:**
- Stable Diffusion, DALL-E 2, Midjourney all use diffusion
- Research focus shifted heavily toward diffusion models
- Active work on making them faster (distillation, better samplers)

**But other models still have niches:**
- VAEs: Great for representation learning, fast prototyping
- GANs: Still used when speed matters (video generation, real-time apps)

## Part 25: Key Takeaways and Summary

### What We Learned

#### 1. Forward Diffusion Process
- Gradually adds Gaussian noise over $T$ timesteps
- Can sample $x_t$ directly from $x_0$ using closed-form formula
- Noise schedule $\beta_t$ controls rate of noise addition
- Signal strength decreases while noise increases over time

#### 2. Reverse Diffusion Process
- Learn to remove noise step-by-step
- Train a U-Net to predict the noise $\epsilon_\theta(x_t, t)$
- Loss is simple: MSE between true noise and predicted noise
- Sampling reverses the process: noise → data

#### 3. U-Net Architecture
- Encoder-decoder with skip connections
- Time embeddings condition on timestep $t$
- Self-attention layers capture long-range dependencies
- Residual blocks with GroupNorm and SiLU activation

#### 4. Sampling Strategies
- **DDPM**: Stochastic, requires all $T$ steps (slow but exact)
- **DDIM**: Deterministic, can skip steps (10-20x faster)
- Trade-off between quality and speed

#### 5. Connection to Score-Based Models
- Predicting noise ≡ Predicting score (gradient of log-probability)
- Multiple equivalent frameworks (DDPM, score matching, SDEs)
- Provides deeper theoretical understanding

### Practical Insights

**Training tips:**
- Start with linear beta schedule, try cosine for better results
- Use U-Net with attention for best quality
- Training is stable - just minimize MSE!
- Monitor generated samples during training

**Sampling tips:**
- Use DDIM for faster sampling during development
- Increase steps for final high-quality samples
- Can use classifier guidance for conditional generation

### Why Diffusion Models Matter

**Revolutionary impact:**
1. **State-of-the-art quality**: Best image generation results
2. **Stable training**: Much easier than GANs
3. **Mode coverage**: Doesn't suffer from mode collapse
4. **Versatility**: Works for images, audio, video, 3D
5. **Conditional generation**: Easy to add text/class conditioning

**Real-world applications:**
- **Text-to-image**: Stable Diffusion, DALL-E 2, Midjourney, Imagen
- **Image editing**: Inpainting, outpainting, style transfer
- **Super-resolution**: Enhance low-res images
- **Audio**: Music generation, text-to-speech
- **Video**: Frame interpolation, video generation

### Further Exploration

**Papers to read:**
1. **DDPM** - "Denoising Diffusion Probabilistic Models" (Ho et al., 2020)
2. **DDIM** - "Denoising Diffusion Implicit Models" (Song et al., 2020)
3. **Score-based** - "Score-Based Generative Modeling through SDEs" (Song et al., 2021)
4. **Stable Diffusion** - "High-Resolution Image Synthesis with Latent Diffusion Models" (Rombach et al., 2022)

**Extensions to try:**
1. **Conditional diffusion**: Add class labels or text embeddings
2. **Classifier-free guidance**: Control generation strength
3. **Latent diffusion**: Run diffusion in VAE latent space (Stable Diffusion approach)
4. **Different datasets**: CIFAR-10, CelebA, custom datasets
5. **Faster samplers**: DPM-Solver, DPM++

### Final Thoughts

Diffusion models represent a paradigm shift in generative modeling:
- **Intuitive**: Progressive denoising is conceptually simple
- **Effective**: State-of-the-art results across domains
- **Flexible**: Easy to extend and modify
- **Growing**: Active research with rapid improvements

The core insight is beautiful: **it's easier to learn many small denoising steps than to generate complex data in one shot!**

Understanding diffusion models is essential for modern AI/ML practitioners - they're not just a research curiosity but the foundation of today's most powerful generative systems.

## Reflection Questions

1. **Understanding**: Can you explain in your own words how the forward and reverse diffusion processes work?

2. **Intuition**: Why is it easier for a neural network to predict noise than to generate images directly?

3. **Trade-offs**: What are the main trade-offs between diffusion models and GANs/VAEs? When would you choose each?

4. **Speed**: Why is DDIM faster than DDPM? What's the key difference in their sampling procedures?

5. **Architecture**: Why do diffusion models use U-Net instead of a simple encoder-decoder? What's special about U-Net?

6. **Applications**: How would you modify this diffusion model to do text-to-image generation (like Stable Diffusion)?

7. **Training**: Why is diffusion model training more stable than GAN training?

8. **Theory**: What's the connection between predicting noise and score-based models? Why are they equivalent?

Take time to think deeply about these questions - understanding the "why" behind diffusion models will help you use and extend them effectively!